In [3]:
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5

Cloning into 'yolov5'...
remote: Enumerating objects: 17847, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 17847 (delta 25), reused 6 (delta 6), pack-reused 17809 (from 3)
Receiving objects: 100% (17847/17847), 16.97 MiB | 2.01 MiB/s, done.
Resolving deltas: 100% (12161/12161), done.
/content/yolov5/yolov5


In [4]:
!pip install -qr requirements.txt
import torch

from IPython.display import Image, clear_output


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 7.8 MB/s eta 0:00:00


In [5]:
import gdown

In [7]:
%cd /content

/content


In [9]:
url = 'https://drive.google.com/file/d/1UlMVv0-k-LpglQOgr7qWctwkJ2COWeQh/view?usp=sharing'
file_id = url.split('/')[-2]

prefix = 'https://drive.google.com/uc?export=download&id='
gdown.download(prefix + file_id)



Downloading...
From (original): https://drive.google.com/uc?export=download&id=1UlMVv0-k-LpglQOgr7qWctwkJ2COWeQh
From (redirected): https://drive.google.com/uc?export=download&id=1UlMVv0-k-LpglQOgr7qWctwkJ2COWeQh&confirm=t&uuid=9fa9a76b-2140-429c-9a5b-de45e99173ba
To: /content/waste_detection.zip
100%|██████████| 40.1M/40.1M [00:00<00:00, 248MB/s]


'waste_detection.zip'

In [10]:
!unzip waste_detection.zip
!rm -f wate_detection.zip

Archive:  waste_detection.zip
   creating: train/
   creating: train/images/
  inflating: train/images/Banana_10_jpg.rf.7c0536bc9eb72ec77c8f515653f44fb9.jpg  
  inflating: train/images/Banana_11_jpg.rf.188551ec3d51326e5562675d285d0e6a.jpg  
  inflating: train/images/Banana_12_jpg.rf.2ecabc9628a9611963e4804d13045377.jpg  
  inflating: train/images/Banana_12_jpg.rf.e951f9b4b66c9896ce6b7a56f9d41438.jpg  
  inflating: train/images/Banana_15_jpg.rf.09c5d34e95e0c06a808a99533e8e737c.jpg  
  inflating: train/images/Banana_15_jpg.rf.c2909c26fa3140f1734b451bead6752b.jpg  
  inflating: train/images/Banana_15_jpg.rf.e7fb3723604fb753bdbca26f382d7aaf.jpg  
  inflating: train/images/Banana_16_jpg.rf.504aa1c1679f9d595e336eba25f6d80c.jpg  
  inflating: train/images/Banana_16_jpg.rf.73833169c2979508c979f27bcb572959.jpg  
  inflating: train/images/Banana_16_jpg.rf.bd09b3cf6ae33d668a421a48b8c4b34b.jpg  
  inflating: train/images/Banana_16_jpg.rf.dbfad378a52e84709bbb248feb3aba2f.jpg  
  inflating: train/im

In [11]:
%cat data.yaml

train: ../train/images
val: ../valid/images


nc: 13
names: ['banana', 'chilli', 'drinkcan', 'drinkpack', 'foodcan', 'lettuce', 'paperbag', 'plasticbag', 'plasticbottle', 'plasticcontainer', 'sweetpotato', 'teabag', 'tissueroll']



#### Define Model Configuration and Architecture

We will write a yaml script that defines parameters for our model like number of classes, anchors, and each layer.

In [12]:
import yaml

with open('data.yaml','r') as stream:
  num_classes = str(yaml.safe_load(stream)['nc'])

In [13]:
num_classes

'13'

In [14]:
# this is the model configuration we will use

%cat /content/yolov5/models/yolov5s.yaml

# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Parameters
nc: 80 # number of classes
depth_multiple: 0.33 # model depth multiple
width_multiple: 0.50 # layer channel multiple
anchors:
  - [10, 13, 16, 30, 33, 23] # P3/8
  - [30, 61, 62, 45, 59, 119] # P4/16
  - [116, 90, 156, 198, 373, 326] # P5/32

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  [
    [-1, 1, Conv, [64, 6, 2, 2]], # 0-P1/2
    [-1, 1, Conv, [128, 3, 2]], # 1-P2/4
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]], # 3-P3/8
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]], # 5-P4/16
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]], # 7-P5/32
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]], # 9
  ]

# YOLOv5 v6.0 head
head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]], # cat backbone P4
    [-1, 3, C3, [512, False]], # 13

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, 

In [21]:
# customize IPython writefile so we can write variables

from IPython.core.magic import register_line_cell_magic

@register_line_cell_magic

def writetemplate(line, cell):
  with open(line, 'w') as f:
    f.write(cell.format(**globals()))

In [28]:
%%writetemplate /content/yolov5/models/custom_yolov5s.yaml

# parameters
# YOLOv5 custom configuration
nc: {num_classes}  # number of classes
depth_multiple: 0.33  # model depth multiple
width_multiple: 0.50  # layer channel multiple

anchors:
  - [10, 13, 16, 30, 33, 23]      # P3/8
  - [30, 61, 62, 45, 59, 119]     # P4/16
  - [116, 90, 156, 198, 373, 326] # P5/32

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  [
   [-1, 1, Conv, [64, 6, 2, 2]],  # 0-P1/2
   [-1, 1, Conv, [128, 3, 2]],   # 1-P2/4
   [-1, 3, C3, [128]],
   [-1, 1, Conv, [256, 3, 2]],   # 3-P3/8
   [-1, 6, C3, [256]],
   [-1, 1, Conv, [512, 3, 2]],   # 5-P4/16
   [-1, 9, C3, [512]],
   [-1, 1, Conv, [1024, 3, 2]],  # 7-P5/32
   [-1, 3, C3, [1024]],
   [-1, 1, SPPF, [1024, 5]],     # 9
  ]

# YOLOv5 v6.0 head
head:
  [
   [-1, 1, Conv, [512, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, "nearest"]],
   [[-1, 6], 1, Concat, [1]],    # cat backbone P4
   [-1, 3, C3, [512, False]],    # 13

   [-1, 1, Conv, [256, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, "nearest"]],
   [[-1, 4], 1, Concat, [1]],    # cat backbone P3
   [-1, 3, C3, [256, False]],    # 17 (P3/8-small)

   [-1, 1, Conv, [256, 3, 2]],
   [[-1, 14], 1, Concat, [1]],   # cat head P4
   [-1, 3, C3, [512, False]],    # 20 (P4/16-medium)

   [-1, 1, Conv, [512, 3, 2]],
   [[-1, 10], 1, Concat, [1]],   # cat head P5
   [-1, 3, C3, [1024, False]],   # 23 (P5/32-large)

   [[17, 20, 23], 1, Detect, [nc, anchors]],  # Detect(P3, P4, P5)
  ]

In [31]:
%cd /content/yolov5/

!python train.py --img 416 --batch 16 --epochs 50 --data '../data.yaml' --cfg ./models/custom_yolov5s.yaml  --weights 'yolov5s.pt' --name yolov5s_results --cache

# increase the epochs size for a good model

Streaming output truncated to the last 5000 lines.
       8/49      1.97G    0.03569     0.0147    0.03538         49        416:  90% 52/58 [00:08<00:00,  7.88it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       8/49      1.97G    0.03574    0.01474    0.03534         51        416:  91% 53/58 [00:08<00:00,  8.19it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       8/49      1.97G    0.03565    0.01474    0.03538         50        416:  93% 54/58 [00:09<00:00,  8.24it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       8/49      1.97G    0.03557    